# Spherical harmonics on the 3-D globe

In [ ]:
%load_ext mcidasv_jupyter
%mcv_connect /path/to/runMcV
%mcv_replay off

## 1. A single spherical harmonic on the globe

In [ ]:
import numpy as np, mcidasv_jupyter as mcv
from scipy.special import lpmv
session = mcv.get_session()

lats = np.linspace(-89, 89, 180)
lons = np.linspace(-179, 179, 360)
theta = np.radians(90 - lats)[:, None]
phi = np.radians(lons)[None, :]
l, m = 6, 4

def real_ylm(l, m, theta, phi, phase=0.0):
    return lpmv(m, l, np.cos(theta)) * np.cos(m * (phi + phase))

field = real_ylm(l, m, theta, phi).astype('f4')

session.run('''
panel = buildWindow(height=600, width=600, panelTypes=GLOBE)
layer = panel[0].createLayer('Color-Shaded Plan View', g)
panel[0].setWireframe(False)
layer.setLayerLabel(label='spherical harmonic Y(6,4)')
''', arrays={'g': (field, lats, lons)})

## 2. Rotate it through time as a movie

In [ ]:
import os
nt = 8
cube = np.stack([real_ylm(l, m, theta, phi, phase=2*np.pi*t/nt)
                 for t in range(nt)]).astype('f4')
OUTDIR = 'output'
os.makedirs(OUTDIR, exist_ok=True)
movie = os.path.join(OUTDIR, 'harmonics_globe.gif')
session.animate_grid(cube, lats, lons, name='Ylm', out=movie, globe=True, fps=6)